# ML-04 — Search Intelligence Data Contract

## 1) Unit of analysis + time window

**Five contract answers:** (1) **Row:** one row in `fact_content_daily_performance` is one pseudonymized content item for one client on one report date; the working frame is one row per `(client_hash_id, content_hash_id)`. (2) **Tables:** use the partitioned `fact_content_daily_performance` for daily performance and `dim_clients` for client history/availability context. (3) **Time window:** March 2026 is the decision snapshot through 2026-03-31; April 1–30, 2026 is the separated outcome window. (4) **Rank/label:** rank items for analyst review and use `declining_clicks_next_30d` as a future click-decline proxy. (5) **Exclude:** exclude April and all other future-derived fields from the honest features because they are unknowable at the March decision moment.

## 2) Fields: feature / label / context / excluded

- **Feature:** March `impressions`, `clicks`, `gsc_avg_position`, `sessions`, and `engagement_rate`, aggregated from daily rows through the decision date. Each is knowable at decision time.
- **Label / proxy:** `declining_clicks_next_30d`, 1 when April clicks are lower than March clicks for the same item. This is a future observed proxy, not a causal claim.
- **Context:** `client_hash_id`, `content_hash_id`, `report_date`, `month`, and `ga4_data_available`; they identify, join, split, or qualify rows but are not model inputs.
- **Excluded:** April metrics and any future-derived field are excluded from the honest feature set because they are not available at the March decision moment. The deliberate-leak column is the label itself and is removed before the honest score.

**Output:** a finite, analyst-facing ranking of items with the strongest measured evidence for review, not an automatic edit decision.


## Setup

This notebook reads remote Parquet with DuckDB and never stores the Hugging Face token. In Colab, put the token in a Secret named `HF_TOKEN`; locally, set the environment variable before starting the kernel. The `_sample` table is not used: it is June 2026, the sealed final month.


In [5]:
%pip -q install duckdb huggingface_hub pandas scikit-learn python-dotenv

import os
from getpass import getpass
from pathlib import Path
import duckdb
import pandas as pd
from sklearn.metrics import roc_auc_score

HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except (ImportError, KeyError):
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass("Hugging Face read token: ")
if not HF_TOKEN:
    raise RuntimeError("A Hugging Face read token is required.")

con = duckdb.connect()
con.execute("INSTALL httpfs")
con.execute("LOAD httpfs")
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")
REL = "hf://datasets/FlyRank/internship-warehouse"
DAILY_MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
DAILY_APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
CONTENT = f"read_parquet('{REL}/dim_content.parquet')"
CLIENTS = f"read_parquet('{REL}/dim_clients.parquet')"
print("Connected to the warehouse; development slice: month=2026-03")

Note: you may need to restart the kernel to use updated packages.
Connected to the warehouse; development slice: month=2026-03


## 3) Three verification queries

These are the only three verification queries. Query 1 checks the stated daily grain. Query 2 checks the March slice count and date span. Query 3 checks availability using `IS TRUE` and reports how many rows survive.


In [6]:
# Query 1 — grain: duplicate daily keys should return zero rows.
q1 = con.sql(f"""
SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS duplicate_rows
FROM {DAILY_MARCH}
GROUP BY client_hash_id, content_hash_id, report_date
HAVING COUNT(*) > 1
LIMIT 5
""").df()
print("Q1 grain duplicate groups:", len(q1))
display(q1)

Q1 grain duplicate groups: 0


,client_hash_id,content_hash_id,report_date,duplicate_rows


In [7]:
# Query 2 — March row count and date span.
q2 = con.sql(f"""
SELECT COUNT(*) AS row_count, MIN(report_date) AS first_date, MAX(report_date) AS last_date
FROM {DAILY_MARCH}
""").df()
display(q2)

,row_count,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [8]:
# Query 3 — availability: explicit IS TRUE, with surviving row count.
q3 = con.sql(f"""
SELECT
  COUNT(*) AS march_rows,
  COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
  COUNT(DISTINCT client_hash_id) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_clients
FROM {DAILY_MARCH}
""").df()
display(q3)

,march_rows,ga4_available_rows,ga4_available_clients
0,9841378,413966,41


## Five-feature frame and availability notes

The frame is deliberately small: exactly five numeric features, aggregated at `(client_hash_id, content_hash_id)` from March. `sessions` and `engagement_rate` use only rows where `ga4_data_available IS TRUE`; unavailable GA4 history is not silently treated as zero.


In [9]:
features = con.sql(f"""
WITH march AS (
  SELECT
    client_hash_id, content_hash_id,
    SUM(gsc_impressions) AS impressions_march,
    SUM(gsc_clicks) AS clicks_march,
    AVG(NULLIF(gsc_avg_position, 0)) AS avg_position_march,
    SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE) AS sessions_march,
    SUM(ga4_engaged_sessions) FILTER (WHERE ga4_data_available IS TRUE) / NULLIF(SUM(ga4_sessions) FILTER (WHERE ga4_data_available IS TRUE), 0) * 100 AS engagement_rate_march
  FROM {DAILY_MARCH}
  GROUP BY client_hash_id, content_hash_id
), april AS (
  SELECT client_hash_id, content_hash_id, SUM(gsc_clicks) AS clicks_april
  FROM {DAILY_APRIL}
  GROUP BY client_hash_id, content_hash_id
)
SELECT m.*, a.clicks_april,
       CAST(COALESCE(a.clicks_april, 0) < m.clicks_march AS INTEGER) AS declining_clicks_next_30d
FROM march m LEFT JOIN april a USING (client_hash_id, content_hash_id)
WHERE m.impressions_march > 0
""").df()
feature_columns = ["impressions_march", "clicks_march", "avg_position_march", "sessions_march", "engagement_rate_march"]
feature_frame = features[["client_hash_id", "content_hash_id", *feature_columns, "declining_clicks_next_30d"]].copy()
display(feature_frame.head())
print(f"Feature frame: {len(feature_frame):,} item rows; exactly {len(feature_columns)} features")

,client_hash_id,content_hash_id,impressions_march,clicks_march,avg_position_march,sessions_march,engagement_rate_march,declining_clicks_next_30d
0,client_fef1a8f436438636,content_c563450113672e49,236.0,0.0,6.986622,NaN,NaN,0
1,client_fef1a8f436438636,content_3d181c429a2d1cf7,1496.0,6.0,4.564125,22.0,0.0,1
2,client_fef1a8f436438636,content_ec486acce8effc3c,1117.0,0.0,4.231107,22.0,0.0,0
3,client_fef1a8f436438636,content_06719aff28a8e977,7083.0,15.0,13.174211,25.0,0.0,1
4,client_fef1a8f436438636,content_593c106c13d1dd80,1071.0,4.0,2.926748,19.0,0.0,1


Feature frame: 176,738 item rows; exactly 5 features


- `impressions_march`: knowable at the decision moment because March Search Console impressions have already been observed.
- `clicks_march`: knowable at the decision moment because March Search Console clicks have already been observed.
- `avg_position_march`: knowable at the decision moment because it is the observed March average position; zero is treated as missing, not as rank zero.
- `sessions_march`: knowable at the decision moment because it sums GA4 sessions only on rows where `ga4_data_available IS TRUE`.
- `engagement_rate_march`: knowable at the decision moment because it averages the observed GA4 rate only where the availability flag is true.

The `declining_clicks_next_30d` column is a label, not a feature: it uses April, which happens after the March decision moment.


## The trap: deliberately add one label-derived column, then remove it

For the trap, the label itself is added as a feature. A one-feature quick score must therefore look perfect (AUC 1.0, subject to a one-class slice). That result is invalid because the label is future information. The honest score uses only the five March features and keeps the label only as the evaluation target.


In [10]:
scored = feature_frame.dropna(subset=feature_columns + ["declining_clicks_next_30d"]).copy()
y = scored["declining_clicks_next_30d"]
if y.nunique() < 2:
    raise ValueError("The March slice has only one label class; the leakage AUC comparison is not defined.")

leaky_score = scored["declining_clicks_next_30d"].astype(float)
honest_score = (
    scored["avg_position_march"].rank(pct=True)
    + scored["clicks_march"].rank(pct=True, ascending=False)
) / 2
leaky_auc = roc_auc_score(y, leaky_score)
honest_auc = roc_auc_score(y, honest_score)
print(f"Deliberate leak AUC (label included as a feature): {leaky_auc:.3f}")
print(f"Honest five-feature directional AUC after removal: {honest_auc:.3f}")
print("Removed before keeping the score: declining_clicks_next_30d")
assert "declining_clicks_next_30d" not in feature_columns
assert leaky_auc >= 0.999


Deliberate leak AUC (label included as a feature): 1.000
Honest five-feature directional AUC after removal: 0.264
Removed before keeping the score: declining_clicks_next_30d


## 4) Limitation

**Named limitation — uneven history and GA4 coverage.** Client histories begin on different dates, and some March rows have `ga4_data_available IS FALSE`. The slice therefore describes the observed March panel, not every possible content item or a balanced population. The ranking cannot establish that an edit caused a later click change, and the April proxy is unavailable at decision time.

## 5) Self-check

- [x] Five plain-words contract answers are stated: row, tables, window, prediction/proxy, exclusion.
- [x] Exactly three verification query cells are present; availability uses `IS TRUE`.
- [x] The feature frame has five features and one availability line per feature.
- [x] The deliberate label-derived leak is shown and removed from the honest feature list.
- [x] The `_sample` final month is excluded from development.
